In [1]:
#%pip install numpy scipy pandas scikit-learn matplotlib wordcloud nltk
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

In [2]:
# Task 1-   Read fetch_20newsgroups from sklearn.datasets.
newsgroups = fetch_20newsgroups(subset='all')
list(newsgroups.target_names)

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [3]:
import nltk
import string
from nltk.stem import PorterStemmer

# Task 2-	Perform necessary preprocessing steps for text data such as
# Task 2a. Tokenization
processed_data = [nltk.word_tokenize(text) for text in newsgroups.data]
print("Data after Tokenization (first example):", processed_data[1])

# Task 2b. Removing all punctuation and lowercase words.
table = str.maketrans('', '', string.punctuation) # make a translation table to remove punctuation
processed_data = [[word.lower().translate(table) for word in text] for text in processed_data]
print("Data after Removing Punctuation and Lowercasing (first example):", processed_data[1])

# Task 2c. Removing the stop words using nltk.
stop_words = set(nltk.corpus.stopwords.words('english'))
processed_data = [[word for word in text if word not in stop_words] for text in processed_data]
print("Data after Removing Stop Words (first example):", processed_data[1])

# Task 2d. Stemming
stemmer = PorterStemmer()
processed_data = [[stemmer.stem(word) for word in text] for text in processed_data]
print("Data after Stemming (first example):", processed_data[1])

# Remove empty strings
processed_data = [[word for word in text if word] for text in processed_data]
print("Data after Removing Empty Strings (first example):", processed_data[1])


Data after Tokenization (first example): ['From', ':', 'mblawson', '@', 'midway.ecn.uoknor.edu', '(', 'Matthew', 'B', 'Lawson', ')', 'Subject', ':', 'Which', 'high-performance', 'VLB', 'video', 'card', '?', 'Summary', ':', 'Seek', 'recommendations', 'for', 'VLB', 'video', 'card', 'Nntp-Posting-Host', ':', 'midway.ecn.uoknor.edu', 'Organization', ':', 'Engineering', 'Computer', 'Network', ',', 'University', 'of', 'Oklahoma', ',', 'Norman', ',', 'OK', ',', 'USA', 'Keywords', ':', 'orchid', ',', 'stealth', ',', 'vlb', 'Lines', ':', '21', 'My', 'brother', 'is', 'in', 'the', 'market', 'for', 'a', 'high-performance', 'video', 'card', 'that', 'supports', 'VESA', 'local', 'bus', 'with', '1-2MB', 'RAM', '.', 'Does', 'anyone', 'have', 'suggestions/ideas', 'on', ':', '-', 'Diamond', 'Stealth', 'Pro', 'Local', 'Bus', '-', 'Orchid', 'Farenheit', '1280', '-', 'ATI', 'Graphics', 'Ultra', 'Pro', '-', 'Any', 'other', 'high-performance', 'VLB', 'card', 'Please', 'post', 'or', 'email', '.', 'Thank', 'you

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# Task 3-	Create bag of words.
# Create the bag of words model
vectorizer = CountVectorizer()
X = vectorizer.fit_transform([' '.join(text) for text in processed_data])

# Display the shape of the resulting bag of words matrix and the first few examples
print("Shape of the bag of words matrix:", X.shape)

# Find the highest frequency words
word_counts = X.sum(axis=0).A1
feature_names = vectorizer.get_feature_names_out()
word_freq = dict(zip(feature_names, word_counts))
sorted_word_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

# Display the top 5 highest frequency words
print("Top 5 highest frequency words:\n")
# Create a dataframe for the highest frequency words
most_freq_words_df = pd.DataFrame(sorted_word_freq[:10], columns=['Word', 'Frequency'])
print(most_freq_words_df)

Shape of the bag of words matrix: (18846, 186159)
Top 5 highest frequency words:

      Word  Frequency
0       ax      61981
1       nt      25825
2     line      21482
3  subject      20675
4    organ      19353
5    would      15874
6      one      15115
7      use      15102
8    write      14848
9   articl      11888


In [5]:
from sklearn.decomposition import LatentDirichletAllocation

# Task 4-	Apply one type of the Topic Modeling methods (LDA or LSA) to find the news topics.
# Applying LDA to find the news topics.
lda = LatentDirichletAllocation(n_components=10, random_state=42)
lda.fit(X)

# Display the top words in each topic
n_top_words = 10
topics = []
for topic_idx, topic in enumerate(lda.components_):
    top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
    top_features = [feature_names[i] for i in top_features_ind]
    topics.append(top_features)
    print(f"Topic #{topic_idx}: {', '.join(top_features)}")


Topic #0: nt, would, one, gun, peopl, go, line, like, write, said
Topic #1: game, nt, line, subject, organ, team, year, write, would, player
Topic #2: ax, max, q3, b8f, a86, 145, 1d9, 0d, pl, 0t
Topic #3: nt, line, organ, subject, use, would, write, get, articl, one
Topic #4: armenian, israel, isra, jew, turkish, arab, muslim, war, state, jewish
Topic #5: line, subject, organ, use, univers, drive, system, email, card, thank
Topic #6: nt, one, would, write, god, peopl, subject, line, articl, say
Topic #7: medic, diseas, patient, health, cancer, use, research, infect, myer, 1993
Topic #8: 25, 10, pt, 11, la, 12, 550, 14, 13, 15
Topic #9: use, file, window, imag, line, program, nt, subject, run, get


In [11]:
from wordcloud import WordCloud
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
import gensim.corpora as corpora


# Task 5-	After finding the “Topics” use word clouds and coherence metrics to modify and get a meaningful set of topics.

# Create a dictionary and corpus for coherence calculation
dictionary = corpora.Dictionary(processed_data)
corpus = [dictionary.doc2bow(text) for text in processed_data]

# Compute coherence score
coherence_model = CoherenceModel(model=lda, texts=processed_data, dictionary=dictionary, coherence='u_mass')
coherence_lda = coherence_model.get_coherence()
print(f'Coherence Score: {coherence_lda}')

# Generate word clouds for each topic
for idx, topic in enumerate(lda.components_):
    wordcloud = WordCloud(stopwords=stop_words, background_color='white').generate_from_frequencies(dict(zip(feature_names, topic)))
    plt.figure()
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(f"Topic #{idx}")
    plt.show()

ModuleNotFoundError: No module named 'gensim.models.coherencemodel'